# Shelter-Cost-to-Income Ratio — Windsor CMA Profiling

**Source table:** Statistics Canada 98-10-0252-01 (Shelter-cost-to-income ratio by tenure)
**Extract:** `data/staging/windsor_shelter_cost.csv`, produced by `etl/extract_windsor_shelter_cost.py`
**Scope:** Windsor (CMA), Ont. only — 2021 Census

This notebook profiles the four assigned dimensions (household income, household type,
tenure, shelter-cost-to-income ratio), documents the two additional dimensions present
in the source (housing suitability, dwelling condition), runs data-quality checks, and
verifies the row-count discrepancy against the project task table (2,389 vs 30,240).
Every claim below was checked against the actual extracted data, not assumed from the
metadata alone — two assumptions made on a first pass turned out to be wrong and are
corrected inline where that happened.

In [1]:
import pandas as pd

STAGING_PATH = "../data/staging/windsor_shelter_cost.csv"

df = pd.read_csv(STAGING_PATH, dtype=str, encoding="utf-8")
print("shape:", df.shape)
df.columns.tolist()

shape: (30240, 16)


['REF_DATE',
 'GEO',
 'DGUID',
 'Household total income groups (14)',
 'Household type including census family structure (16)',
 'Housing suitability (3)',
 'Dwelling condition (3)',
 'Statistics (3C)',
 'Shelter-cost-to-income ratio (5)',
 'Coordinate',
 'Tenure (3):Total - Tenure[1]',
 'Symbol',
 'Tenure (3):Owner[2]',
 'Symbol.1',
 'Tenure (3):Renter[3]',
 'Symbol.2']

## Phase 2 — Profiling the four assigned dimensions

Three of the four assigned dimensions (household income, household type, ratio) are
**row-level categorical dimensions** — each row carries one category label per column.
**Tenure is structurally different**: in this export it is **pivoted into three columns**
(`Tenure (3):Total - Tenure[1]`, `Tenure (3):Owner[2]`, `Tenure (3):Renter[3]`), each with
its own `Symbol` column, rather than appearing as category labels in a single column.

In [2]:
income_col = "Household total income groups (14)"
counts = df[income_col].value_counts(dropna=False)
print(f"{len(counts)} distinct categories observed\n")
print(counts.to_string())

14 distinct categories observed

Household total income groups (14)
Total - Total income of household        2160
Under $10,000                            2160
$10,000 to $19,999                       2160
$20,000 to $29,999                       2160
$30,000 to $39,999                       2160
$40,000 to $49,999                       2160
$50,000 to $59,999                       2160
$60,000 to $69,999                       2160
$70,000 to $79,999                       2160
$80,000 to $89,999                       2160
$90,000 to $99,999                       2160
$100,000 and over                        2160
Median total income of household ($)     2160
Average total income of household ($)    2160


In [3]:
is_total = df[income_col].str.startswith("Total -", na=False)
print("Total-category rows:", is_total.sum())
print("Detailed-category rows:", (~is_total).sum())
print("\nDetailed categories:")
print(df.loc[~is_total, income_col].unique())

Total-category rows: 2160
Detailed-category rows: 28080

Detailed categories:
<StringArray>
[                        'Under $10,000',
                    '$10,000 to $19,999',
                    '$20,000 to $29,999',
                    '$30,000 to $39,999',
                    '$40,000 to $49,999',
                    '$50,000 to $59,999',
                    '$60,000 to $69,999',
                    '$70,000 to $79,999',
                    '$80,000 to $89,999',
                    '$90,000 to $99,999',
                     '$100,000 and over',
  'Median total income of household ($)',
 'Average total income of household ($)']
Length: 13, dtype: str


In [4]:
household_col = "Household type including census family structure (16)"
counts = df[household_col].value_counts(dropna=False)
print(f"{len(counts)} distinct TEXT labels observed\n")
print(counts.to_string())

14 distinct TEXT labels observed

Household type including census family structure (16)
Without children                                                               3780
With children                                                                  3780
Total - Household type including census family structure                       1890
Census-family households                                                       1890
One-census-family households without additional person                         1890
One couple census family without other persons in the household                1890
One-census-family household without additional persons: one-parent family      1890
One-census-family households with additional persons                           1890
One couple census family with other persons in the household                   1890
One-census-family household with additional persons: one-parent family         1890
Multiple-census-family household                                        

**Label collision found.** The metadata declares 16 distinct household-type
*member IDs*, but the text above shows only 14 distinct *labels* — "Without children"
and "With children" each appear twice, with 3,780 rows (= 2 x 1,890) instead of 1,890.
This isn't duplicate data: the `Coordinate` column encodes the StatCan member ID
positionally, and it shows these labels genuinely belong to two different household-type
branches (e.g. "couple family without additional persons" vs "couple family with
additional persons") that happen to render with identical child-presence wording.
Any analysis that groups by this column's text alone will silently merge two distinct
categories — grouping must use `Coordinate` or a combination of columns to disambiguate.

In [5]:
wc = df[df[household_col] == "Without children"][["Coordinate"]].copy()
wc["household_type_member_id"] = wc["Coordinate"].str.split(".").str[2]
print("Member IDs sharing the label 'Without children':", sorted(wc["household_type_member_id"].unique(), key=int))

all_member_ids = df["Coordinate"].str.split(".").str[2]
print("Total distinct household-type member IDs in the file (via Coordinate):", all_member_ids.nunique())
print("-> Matches the metadata's declared 16 household-type members:", all_member_ids.nunique() == 16)

Member IDs sharing the label 'Without children': ['5', '10']
Total distinct household-type member IDs in the file (via Coordinate): 16
-> Matches the metadata's declared 16 household-type members: True


In [6]:
is_total = df[household_col].str.startswith("Total -", na=False)
print("Total-category rows:", is_total.sum())
print("Detailed-category rows (14 distinct labels, 16 real categories):", (~is_total).sum())

Total-category rows: 1890
Detailed-category rows (14 distinct labels, 16 real categories): 28350


In [7]:
ratio_col = "Shelter-cost-to-income ratio (5)"
counts = df[ratio_col].value_counts(dropna=False)
print(f"{len(counts)} distinct categories observed\n")
print(counts.to_string())

5 distinct categories observed

Shelter-cost-to-income ratio (5)
Total - Shelter-cost-to-income ratio                 6048
Spending less than 30% of income on shelter costs    6048
Spending 30% or more of income on shelter costs      6048
30% to less than 100%                                6048
Not applicable                                       6048


In [8]:
is_total = df[ratio_col].str.startswith("Total -", na=False)
print("Total-category rows:", is_total.sum())
print("Detailed-category rows:", (~is_total).sum())
print("\nDetailed categories:")
print(df.loc[~is_total, ratio_col].unique())

Total-category rows: 6048
Detailed-category rows: 24192

Detailed categories:
<StringArray>
['Spending less than 30% of income on shelter costs',
   'Spending 30% or more of income on shelter costs',
                             '30% to less than 100%',
                                    'Not applicable']
Length: 4, dtype: str


### Tenure (pivoted into columns, not rows)

Tenure has 3 published categories (Total - Tenure, Owner, Renter). Each is a separate
observation column holding a count of private households, paired with its own `Symbol`
column for data-quality/suppression flags. There is no single "Tenure category" column
to tally — instead we report non-null / non-suppressed coverage per tenure column.

In [9]:
tenure_cols = [
    "Tenure (3):Total - Tenure[1]",
    "Tenure (3):Owner[2]",
    "Tenure (3):Renter[3]",
]
symbol_cols = ["Symbol", "Symbol.1", "Symbol.2"]

for tcol, scol in zip(tenure_cols, symbol_cols):
    non_null = df[tcol].notna().sum()
    null = df[tcol].isna().sum()
    symbol_present = df[scol].notna().sum()
    print(f"{tcol}")
    print(f"  non-null observation values: {non_null}")
    print(f"  null/blank observation values: {null}")
    print(f"  rows with a symbol flag ({scol}): {symbol_present}")
    if symbol_present:
        print(f"  symbol values used: {sorted(df.loc[df[scol].notna(), scol].unique())}")
    print()

Tenure (3):Total - Tenure[1]
  non-null observation values: 29940
  null/blank observation values: 300
  rows with a symbol flag (Symbol): 1128
  symbol values used: ['...', 'x']

Tenure (3):Owner[2]
  non-null observation values: 30030
  null/blank observation values: 210
  rows with a symbol flag (Symbol.1): 1404
  symbol values used: ['...', 'x']

Tenure (3):Renter[3]
  non-null observation values: 29844
  null/blank observation values: 396
  rows with a symbol flag (Symbol.2): 1314
  symbol values used: ['...', 'x']



## Additional dimensions present in the source (not part of the four assigned)

The source table carries two more dimensions — **Housing suitability** and
**Dwelling condition** — plus a **Statistics** dimension (point estimate vs confidence
interval bounds). These are not part of the four assigned dimensions, but they are the
reason the Windsor extract has far more rows than a simple income x household-type x
ratio table would.

In [10]:
for col in ["Housing suitability (3)", "Dwelling condition (3)", "Statistics (3C)"]:
    counts = df[col].value_counts(dropna=False)
    print(f"--- {col} ({len(counts)} categories) ---")
    print(counts.to_string())
    print()

--- Housing suitability (3) (3 categories) ---
Housing suitability (3)
Total - Housing suitability    10080
Suitable                       10080
Not suitable                   10080

--- Dwelling condition (3) (3 categories) ---
Dwelling condition (3)
Total - Dwelling condition                            10080
Only regular maintenance  and minor repairs needed    10080
Major repairs needed                                  10080

--- Statistics (3C) (3 categories) ---
Statistics (3C)
Number of private households                                       10080
Confidence interval lower bound of number of private households    10080
Confidence interval upper bound of number of private households    10080



**Do these affect the requested analysis?** Yes, materially:

- `Housing suitability` and `Dwelling condition` each add a Total + 2 detailed categories.
  If the assigned analysis only needs overall income/household-type/ratio patterns, rows
  should be filtered to `Total - Housing suitability` and `Total - Dwelling condition`
  to avoid treating suitability/condition breakdowns as additional, separate households.
- `Statistics (3C)` carries 3 variants per combination: the point estimate
  ("Number of private households") plus a 95% confidence-interval lower and upper bound.
  Only the point estimate should be used for counts/KPIs; the CI bound rows are not
  additional households and must not be summed with the point estimate.
- The `Household total income groups` dimension itself mixes **counts** (the 12 bracket
  categories) with **dollar statistics** ("Median total income of household ($)" and
  "Average total income of household ($)") that are not household counts at all — see
  the reconciliation section below for why this matters.

## Phase 3 — Data-quality checks

### Row counts (source vs. extracted)

Verified independently by `etl/extract_windsor_shelter_cost.py` while streaming the full
1.28 GB source file in chunks: **5,019,840** total source rows across **166** geographies,
of which **30,240** matched `GEO == "Windsor (CMA), Ont."`. Re-confirmed here from the
staged output file:

In [11]:
print("Extracted (staging) row count:", len(df))
print("Extracted file column count:", df.shape[1])
print("Distinct GEO values in staging file:", df['GEO'].unique())
print("Distinct DGUID values in staging file:", df['DGUID'].unique())

Extracted (staging) row count: 30240
Extracted file column count: 16
Distinct GEO values in staging file: <StringArray>
['Windsor (CMA), Ont.']
Length: 1, dtype: str
Distinct DGUID values in staging file: <StringArray>
['2021S0503559']
Length: 1, dtype: str


### Missing / blank values by column

In [12]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("No missing (NaN/blank) values in any column.")
else:
    print(missing.to_string())

Symbol                          29112
Symbol.2                        28926
Symbol.1                        28836
Tenure (3):Renter[3]              396
Tenure (3):Total - Tenure[1]      300
Tenure (3):Owner[2]               210


### Special StatCan symbols (not treated as missing or zero)

Per the metadata's Symbol Legend: `x` = suppressed for confidentiality, `F` = too
unreliable to publish, `..` = not available, `...` = not applicable, `E` = use with
caution, `r` = revised, `p` = preliminary. These appear in the `Symbol*` columns next to
each tenure observation, flagging that specific observation — they are not blank values
and are not treated as zero anywhere in this notebook.

In [13]:
for scol in symbol_cols:
    vc = df[scol].value_counts(dropna=True)
    print(f"{scol}: {vc.to_dict() if not vc.empty else 'no symbols present'}")

Symbol: {'...': 828, 'x': 300}
Symbol.1: {'...': 1194, 'x': 210}
Symbol.2: {'...': 918, 'x': 396}


In [14]:
# Check whether any tenure observation columns contain a non-numeric placeholder
# (e.g. 'x') directly in the value column itself, rather than a number.
import re
numeric_pattern = re.compile(r"^-?\d+$")
for tcol in tenure_cols:
    non_numeric = df[tcol].dropna()[~df[tcol].dropna().str.match(numeric_pattern)]
    print(f"{tcol}: {len(non_numeric)} non-numeric observation values",
          sorted(non_numeric.unique()) if len(non_numeric) else "")

Tenure (3):Total - Tenure[1]: 0 non-numeric observation values 


Tenure (3):Owner[2]: 0 non-numeric observation values

 
Tenure (3):Renter[3]: 0 non-numeric observation values 


### Duplicate records

**First attempt (wrong key):** building a composite key from the text category columns
(income, household type, suitability, dwelling condition, statistics, ratio) flagged
7,560 "duplicate" rows — but this is a false signal caused by the household-type label
collision documented above: two genuinely different categories share the same text, so a
text-based key merges them. **Corrected approach:** use the `Coordinate` column, which
StatCan generates from the actual dimension member IDs and is the correct uniqueness key
for this table.

In [15]:
print("Unique Coordinate values:", df['Coordinate'].nunique(), "of", len(df), "rows")
print("Duplicate rows by Coordinate:", df.duplicated(subset=['Coordinate']).sum())

expected_rows = 14 * 16 * 3 * 3 * 3 * 5
print()
print("Expected unique combinations (14 income x 16 household-type x 3 suitability x 3 dwelling x 3 statistics x 5 ratio):", expected_rows)
print("Actual rows:", len(df))
print("Match:", expected_rows == len(df))

Unique Coordinate values: 30240 of 30240 rows


Duplicate rows by Coordinate: 0

Expected unique combinations (14 income x 16 household-type x 3 suitability x 3 dwelling x 3 statistics x 5 ratio): 30240
Actual rows: 30240
Match: True


**No duplicate records exist.** Every row has a unique `Coordinate`, and the count
matches the full dimensional cross-product exactly. The earlier text-key check is kept
above (not deleted) because the label-collision finding it surfaced is itself an
important data-quality note for anyone grouping by the household-type column downstream.

### Total/subtotal reconciliation

Rule followed throughout: compare a *subtotal breakdown* against its *stated total* —
never sum a total row together with its own subtotal rows. The income dimension's
"Median"/"Average" rows are dollar statistics, not household counts, and are excluded
from count-sum reconciliation since medians and averages of subgroups do not sum to the
total's median/average by definition.

In [16]:
# Tenure: Owner + Renter vs Total-Tenure, point-estimate statistic only,
# excluding Median/Average income rows (not counts) and suppressed/non-numeric values.
point_estimate = df[df["Statistics (3C)"] == "Number of private households"].copy()
count_rows = point_estimate[~point_estimate[income_col].isin([
    "Median total income of household ($)", "Average total income of household ($)",
])].copy()

for tcol in tenure_cols:
    count_rows[tcol + "_num"] = pd.to_numeric(count_rows[tcol], errors="coerce")

clean = count_rows.dropna(subset=[c + "_num" for c in tenure_cols])
diff = (clean["Tenure (3):Owner[2]_num"] + clean["Tenure (3):Renter[3]_num"]) - clean["Tenure (3):Total - Tenure[1]_num"]
mismatch = diff[diff != 0]

print(f"Household-count rows checked (Median/Average + suppressed excluded): {len(clean)} of {len(count_rows)}")
print(f"Exact reconciliation (Owner + Renter == Total): {(diff == 0).sum()} rows")
print(f"Non-zero difference: {len(mismatch)} rows")
print(f"Magnitude of non-zero differences -> min: {mismatch.min()}, max: {mismatch.max()}")
print()
print("Distribution of |difference| for mismatched rows:")
print(mismatch.abs().value_counts().sort_index().head(10).to_string())

Household-count rows checked (Median/Average + suppressed excluded): 8640 of 8640
Exact reconciliation (Owner + Renter == Total): 6123 rows
Non-zero difference: 2517 rows
Magnitude of non-zero differences -> min: -35, max: 10

Distribution of |difference| for mismatched rows:


5     1930
10     277
15     193
20      77
25      34
30       5
35       1


**Interpretation:** all non-zero differences are small (|diff| <= 35, the large
majority at exactly 5 or 10), consistent with Statistics Canada's **random rounding**
disclosure-control practice applied to small census counts (component cells are rounded
independently, so they don't always sum exactly to a published total). This is expected
StatCan behavior for this kind of table, not a defect in the extraction — but it was
verified by checking the actual magnitude/distribution of differences rather than assumed.
A first pass that mistakenly included the Median/Average income rows showed mismatches up
to 410,000 — entirely an artifact of summing dollar-value statistics as if they were
counts, not a real reconciliation failure; excluding those rows (as done above) was
necessary before this check meant anything.

In [17]:
# Ratio: <30% + >=30% + Not applicable vs Total, for a simple baseline slice
# (Total income, Total household type, Total suitability, Total dwelling condition),
# point-estimate statistic, Total-Tenure column only.
baseline = count_rows[
    (count_rows[income_col] == "Total - Total income of household")
    & (count_rows[household_col] == "Total - Household type including census family structure")
    & (count_rows["Housing suitability (3)"] == "Total - Housing suitability")
    & (count_rows["Dwelling condition (3)"] == "Total - Dwelling condition")
].copy()
baseline["Tenure (3):Total - Tenure[1]_num"] = pd.to_numeric(baseline["Tenure (3):Total - Tenure[1]"], errors="coerce")

baseline_by_ratio = baseline.set_index(ratio_col)["Tenure (3):Total - Tenure[1]_num"]
print(baseline_by_ratio.to_string())

total = baseline_by_ratio.get("Total - Shelter-cost-to-income ratio")
under30 = baseline_by_ratio.get("Spending less than 30% of income on shelter costs")
over30 = baseline_by_ratio.get("Spending 30% or more of income on shelter costs")
not_applicable = baseline_by_ratio.get("Not applicable")
partial_30_100 = baseline_by_ratio.get("30% to less than 100%")

print()
print("Total:", total)
print("under30 + over30 + not_applicable =", under30 + over30 + not_applicable,
      "-> matches Total:", (under30 + over30 + not_applicable) == total)
print("'30% to less than 100%' vs its stated parent '>=30%':", partial_30_100, "vs", over30,
      "-> equal:", partial_30_100 == over30)

Shelter-cost-to-income ratio (5)
Total - Shelter-cost-to-income ratio                 164730
Spending less than 30% of income on shelter costs    135475
Spending 30% or more of income on shelter costs       28745
30% to less than 100%                                 26210
Not applicable                                          510

Total: 164730
under30 + over30 + not_applicable = 164730 -> matches Total: True
'30% to less than 100%' vs its stated parent '>=30%': 26210 vs 28745 -> equal: False


**Finding:** `Total = <30% + >=30% + Not applicable` reconciles exactly for this
baseline (Windsor-wide, all-household) slice. However, `30% to less than 100%` (member 4,
a child of member 3 per the metadata hierarchy) does **not** equal its stated parent
`>=30%` (member 3) — it is a partial breakdown with no sibling category published to
account for the remainder (e.g. households spending 100%+ of income on shelter). This is
flagged as an open question below rather than assumed away.

## Phase 4 — Row-count discrepancy: 2,389 (proposal) vs 30,240 (this extract)

The project proposal's task table cites **2,389 rows** for the Windsor CMA file; our
verified extract has **30,240 rows**. Checked against the actual data (via `Coordinate`
member IDs, not just the metadata's declared counts, since the household-type column's
text labels undercounted at 14 due to the label collision found above):

In [18]:
dims_text_nunique = {
    "Household total income groups": income_col,
    "Household type (by TEXT label -- undercounts due to collision)": household_col,
    "Housing suitability": "Housing suitability (3)",
    "Dwelling condition": "Dwelling condition (3)",
    "Statistics": "Statistics (3C)",
    "Shelter-cost-to-income ratio": ratio_col,
}
for label, col in dims_text_nunique.items():
    print(f"{label}: {df[col].nunique()} distinct text values")

print()

coord_segments = df["Coordinate"].str.split(".", expand=True)
# segment order (StatCan Dimension ID order, geography excluded since file is Windsor-only):
# 0=Geography 1=Income 2=HouseholdType 3=Suitability 4=DwellingCondition 5=Statistics 6=Ratio
dims_by_member_id = {
    "Household total income groups": 1,
    "Household type (by real member ID)": 2,
    "Housing suitability": 3,
    "Dwelling condition": 4,
    "Statistics": 5,
    "Shelter-cost-to-income ratio": 6,
}
product = 1
for label, seg_idx in dims_by_member_id.items():
    n = coord_segments[seg_idx].nunique()
    product *= n
    print(f"{label}: {n} distinct member IDs")

print()
print("Product of all real dimension member counts:", product)
print("Actual extracted row count:", len(df))
print("Match:", product == len(df))

Household total income groups: 14 distinct text values


Household type (by TEXT label -- undercounts due to collision): 14 distinct text values
Housing suitability: 3 distinct text values
Dwelling condition: 3 distinct text values
Statistics: 3 distinct text values
Shelter-cost-to-income ratio: 5 distinct text values

Household total income groups: 14 distinct member IDs
Household type (by real member ID): 16 distinct member IDs
Housing suitability: 3 distinct member IDs
Dwelling condition: 3 distinct member IDs
Statistics: 3 distinct member IDs
Shelter-cost-to-income ratio: 5 distinct member IDs

Product of all real dimension member counts: 30240
Actual extracted row count: 30240
Match: True


**Conclusion:** 30,240 = 14 x 16 x 3 x 3 x 3 x 5 — the full cross-product of every
*real* category (verified via `Coordinate` member IDs) in income, household type, housing
suitability, dwelling condition, statistics and ratio dimensions, with tenure pivoted
into columns rather than rows. A naive check using the household-type column's text
labels would have found only 14 x 14 x 3 x 3 x 3 x 5 = 26,460 and wrongly concluded the
cross-product didn't match the row count — it does, once the label collision is accounted
for. The 2,389 figure in the proposal was very likely produced from StatCan's web table
interface with a narrower selection (e.g., only the point-estimate statistic and/or
Total-only suitability and dwelling condition), not from this full bulk CSV export. This
explanation is supported directly by the data; the original 2,389 source/selection was
not available to confirm the exact filters used, so it remains a well-supported inference
rather than a fully closed question.

## Summary

- Extract verified independently: 30,240 Windsor (CMA), Ont. rows, 16 columns, single
  consistent DGUID (`2021S0503559`); source file untouched (1,376,774,682 bytes).
- Four assigned dimensions profiled with exact published category labels and counts;
  Tenure's column-based (not row-based) structure documented.
- **Household-type label collision found**: two member IDs render as "Without children"
  and two as "With children" across different branches — text-based grouping on this
  column will silently merge distinct categories; `Coordinate` disambiguates correctly.
- Housing suitability, dwelling condition and statistics identified as the additional
  dimensions inflating row count, with guidance on filtering them for a simpler analysis.
- No missing/blank values found in the checked columns; StatCan symbol/suppression values
  documented separately per the metadata's Symbol Legend and never treated as zero.
- No duplicate records when keyed correctly by `Coordinate` (0 duplicates, exact match to
  expected cross-product); an initial text-key check had produced a false "7,560
  duplicates" result, which is explained rather than hidden.
- Tenure total/subtotal reconciles exactly for the large majority of household-count
  rows checked (exact counts printed in the reconciliation cell above); all non-zero
  differences are small (|diff| <= 35) and consistent with StatCan's random-rounding
  disclosure control, not a data error.
- Ratio total reconciles exactly for the Windsor-wide baseline across its three top-level
  categories; the `30% to less than 100%` sub-category does not fully account for its
  stated parent `>=30%` — flagged as an open question, not forced to reconcile.
- Row-count discrepancy (2,389 vs 30,240) fully explained by the real dimensional
  cross-product, confirmed via `Coordinate` member IDs against the actual data.